In [1]:
import os
import sys

# ============================================================================
# ENVIRONMENT SETUP & PATH RESOLUTION | Optional
# ============================================================================
# Forces Python to look into the local virtual environment's site-packages first.
# This prevents IDE/Jupyter kernel path mismatch issues with installed libraries.
venv_path = os.path.join(os.getcwd(), ".venv", "Lib", "site-packages")
if os.path.exists(venv_path) and venv_path not in sys.path:
    sys.path.insert(0, venv_path)

# Verify core dependencies before starting execution
try:
    import faiss
    from langchain_community.retrievers import BM25Retriever
    print("🎉 Success! Core retrieval dependencies (FAISS, BM25) are verified.")
except ImportError as e:
    print(f"⚠️ Critical Dependency Missing: {e}")
    sys.exit(1)

C:\Users\shiva\AppData\Local\Temp\ipykernel_20508\3851538691.py:16: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


🎉 Success! Core retrieval dependencies (FAISS, BM25) are verified.


In [17]:
#LangChain imports
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain.chat_models import init_chat_model
from langchain_classic.chains import create_retrieval_chain

# Document Loaders
from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader,
    PyPDFLoader,
    Docx2txtLoader
)

In [4]:
# ============================================================================
# STEP 1: DOCUMENT LOADING
# ============================================================================
# Load raw text documents from the temporary directory path
data_directory = r"C:\Users\shiva\AppData\Local\Temp\tmp8qb1tq5f"

print(f"\nScanning and loading text files from: {data_directory}...")
loader = DirectoryLoader(
    path=data_directory,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)

documents = loader.load()
print(f"Successfully loaded {len(documents)} source document(s).")


Scanning and loading text files from: C:\Users\shiva\AppData\Local\Temp\tmp8qb1tq5f...
Successfully loaded 3 source document(s).


In [18]:
# ============================================================================
# STEP 1: DEFINE IN-MEMORY DOCUMENTS (UPDATED)
# ============================================================================
# Your new documents are declared directly as a Python list here
documents = [
    Document(page_content="LangChain helps build LLM applications.", metadata={"source": "custom_list"}),
    Document(page_content="Pinecone is a vector database for semantic search.", metadata={"source": "custom_list"}),
    Document(page_content="The Eiffel Tower is located in Paris.", metadata={"source": "custom_list"}),
    Document(page_content="Langchain can be used to develop agentic ai application.", metadata={"source": "custom_list"}),
    Document(page_content="Langchain has many types of retrievers.", metadata={"source": "custom_list"})
]
print(f"Successfully loaded {len(documents)} in-memory source document(s).")

Successfully loaded 5 in-memory source document(s).


In [19]:
# ============================================================================
# STEP 2: TEXT SPLITTING & CHUNKING
# ============================================================================
# RecursiveCharacterTextSplitter cleanly splits large files down into manageable sizes
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len
)

chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} structural chunks from raw documents.")

Created 5 structural chunks from raw documents.


In [20]:
# ============================================================================
# STEP 3: EMBEDDINGS ENGINE INITIALIZATION
# ============================================================================
# Uses OpenAI's text-embedding-ada-002 model by default to convert text into math vectors
# (Ensure your OPENAI_API_KEY environment variable is set before running)
embedding_model = OpenAIEmbeddings()

In [21]:
# ============================================================================
# STEP 4: VECTOR STORE SETUP (DENSE RETRIEVAL LAYER)
# ============================================================================
# Initialize and build a local directory-persisted Chroma vector store index
persist_dir = "./chroma_db"
print(f"Indexing chunks inside ChromaDB vector store at: {persist_dir}...")

vectorstore = Chroma.from_documents(
    documents=chunks,  # Passing split chunks for high semantic alignment
    embedding=embedding_model,
    collection_name="RAG-Collection",
    persist_directory=persist_dir
)

# Convert vectorstore into a basic Dense semantic search retriever
dense_retriever = vectorstore.as_retriever()

Indexing chunks inside ChromaDB vector store at: ./chroma_db...


In [22]:
# ============================================================================
# STEP 5: BM25 SEARCH SETUP (SPARSE RETRIEVAL LAYER)
# ============================================================================
# Initialize keyword matcher using BM25 over the exact same chunk slices
print("Initializing Sparse BM25 Keyword Search Index...")
sparse_retriever = BM25Retriever.from_documents(chunks)
sparse_retriever.k = 3  # Retrieve the top 3 keyword-matched document layers

Initializing Sparse BM25 Keyword Search Index...


In [23]:
# ============================================================================
# STEP 6: ENSEMBLE HYBRID ENGINE COMPILATION
# ============================================================================
# Melds semantic context lookup (Chroma) and precise keywords (BM25) using RRF
# Weights give 70% importance to conceptual context, 30% to word matching
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7, 0.3]
)
print("Hybrid Ensemble search algorithm bound successfully.")

Hybrid Ensemble search algorithm bound successfully.


In [24]:
# ============================================================================
# STEP 7: PROMPT & LLM DEFINITIONS
# ============================================================================
system_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""

prompt = ChatPromptTemplate([
    ("system", system_prompt),
    ("human", "{input}")
])

# Initialize the chat model using the unified standard factory pattern
llm = init_chat_model("openai:gpt-3.5-turbo")

# Create the standard multi-document synthesis formatting chain layer
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)

In [25]:
# ============================================================================
# STEP 8: FINAL LCEL RAG CHAIN COMPILATION
# ============================================================================
# Ties the hybrid document finder to the synthesis LLM engine layout
rag_chain = create_retrieval_chain(
    retriever=hybrid_retriever, 
    combine_docs_chain=document_chain
)

In [26]:
# ============================================================================
# STEP 9: EXECUTION & TEST QUERY RUNNER
# ============================================================================
query = {"input": "How can I build an app using LLMs?"}
print(f"\nInvoking Hybrid RAG Pipeline for query: '{query['input']}'...\n")

response = rag_chain.invoke(query)


Invoking Hybrid RAG Pipeline for query: 'How can I build an app using LLMs?'...



In [27]:
# Display synthesized output answer
print("=" * 60)
print("✅ Answer:")
print(response["answer"])
print("=" * 60)

# Display source tracing reference documents used by the chain
print("\n📄 Source Documents Utilized:")
for i, doc in enumerate(response["context"]):
    # Pull file identifier cleanly from absolute metadata paths
    source_name = os.path.basename(doc.metadata.get('source', 'Unknown'))
    print(f"\n[Doc {i+1}] Source: {source_name}")
    print(f"Content snippet: {doc.page_content.strip()[:200]}...")

✅ Answer:
You can build an app using LLMs by utilizing tools like LangChain specifically designed for building LLM applications. LangChain provides support for developing applications with large language models and offers various types of retrievers to enhance functionality. Utilizing LangChain can help streamline the process of building apps that leverage LLM technology.

📄 Source Documents Utilized:

[Doc 1] Source: custom_list
Content snippet: LangChain helps build LLM applications....

[Doc 2] Source: custom_list
Content snippet: Langchain can be used to develop agentic ai application....

[Doc 3] Source: doc_0.txt
Content snippet: Machine Learning Fundamentals...

[Doc 4] Source: doc_1.txt
Content snippet: Deep Learning and Neural Networks...

[Doc 5] Source: custom_list
Content snippet: Langchain has many types of retrievers....
